In [ ]:
import sqlite3
import pandas as pd
import os
from loguru import logger

# Configuratie: Pad naar de hoofddatabase
SDM_PAD = 'SDM.db'

def reset_sdm():
    """Maakt alle tabellen in de SDM database leeg."""
    if not os.path.exists(SDM_PAD):
        logger.warning(f"Reset overgeslagen: {SDM_PAD} bestaat nog niet.")
        return

    try:
        with sqlite3.connect(SDM_PAD) as conn:
            cursor = conn.cursor()
            # Haal alle tabelnamen op
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
            tabellen = [t[0] for t in cursor.fetchall()]

            for tabel in tabellen:
                cursor.execute(f"DELETE FROM {tabel}")
            
            conn.commit()
            # Optioneel: ruim de database op (maakt bestand kleiner)
            cursor.execute("VACUUM")
            
        logger.success("Reset geslaagd! Alle tabellen in SDM zijn leeg.")
    except Exception as e:
        logger.error(f"Fout bij reset: {e}")

def overzetten_alle_data():
    logger.info("Start van de volledige data-pomp naar SDM...")

    mapping_config = {
        'BikeToDrive_1_Accessoireverkoop.db': {
            'Klant': 'Accessoire_Verkoop_Klant', 'Filiaal': 'Accessoire_Verkoop_Filiaal',
            'Monteur': 'Accessoire_Verkoop_Monteur', 'Leverancier': 'Accessoire_Verkoop_Leverancier',
            'Accessoire': 'Accessoire_Verkoop_Accessoire', 'Accessoire_Verkoop': 'Accessoire_Verkoop'
        },
        'BikeToDrive_2_Fietsverkoop.db': {
            'Klant': 'Fietsverkoop_Klant', 'Filiaal': 'Fietsverkoop_Filiaal',
            'Fiets': 'Fietsverkoop_Fiets', 'Fiets_Verkoop': 'Fiets_Verkoop'
        },
        'BikeToDrive_3_Onderhoud.db': {
            'Fabrikant': 'Onderhoud_Fabrikant', 'Fiets': 'Onderhoud_Fiets',
            'Filiaal': 'Onderhoud_Filiaal', 'Monteur': 'Onderhoud_Monteur', 'Onderhoud': 'Onderhoud'
        },
        'BikeToDrive_4_Accessoire_Inkoop.db': {
            'Leverancier': 'Accessoire_Inkoop_Leverancier', 'Accessoire': 'Accessoire_Inkoop_Accessoire',
            'Accessoire_Inkoop': 'Accessoire_Inkoop'
        },
        'BikeToDrive_5_Fiets_Inkoop.db': {
            'Fabrikant': 'Fiets_Inkoop_Fabrikant', 'Fiets': 'Fiets_Inkoop_Fiets',
            'Fiets_Inkoop': 'Fiets_Inkoop'
        }
    }

    try:
        # Open de doel-database één keer voor het hele proces
        with sqlite3.connect(SDM_PAD) as conn_sdm:
            for bron_db, tabellen_map in mapping_config.items():
                if not os.path.exists(bron_db):
                    logger.error(f"Bestand niet gevonden: {bron_db}")
                    continue
                
                logger.debug(f"Verwerken van: {bron_db}")
                
                # Open bron-database
                with sqlite3.connect(bron_db) as conn_bron:
                    for bron_tabel, sdm_tabel in tabellen_map.items():
                        try:
                            df = pd.read_sql_query(f"SELECT * FROM {bron_tabel}", conn_bron)
                            
                            if not df.empty:
                                df.to_sql(sdm_tabel, conn_sdm, if_exists='append', index=False)
                                logger.success(f"  [ OK ] {len(df)} rijen -> {sdm_tabel}")
                            else:
                                logger.warning(f"  [LEEG] {bron_tabel} bevat geen data.")
                                
                        except Exception as e:
                            logger.warning(f"  [FOUT] Kon {bron_tabel} niet overzetten: {e}")

        logger.success("Klaar! Het ETL-proces is succesvol afgerond.")

    except Exception as e:
        logger.critical(f"Het proces is gestopt door een fatale fout: {e}")

# Uitvoering
if __name__ == "__main__":
    reset_sdm()
    overzetten_alle_data()